In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RUN = Path("results/runs/latest")          # the symlink you set up

solves      = pd.read_csv(RUN / "solves.csv")
joints      = pd.read_csv(RUN / "joints.csv")
constraints = pd.read_csv(RUN / "constraints.csv")
timings     = pd.read_csv(RUN / "timings.csv")

print(f"{len(solves)} solve rows | solvers: {sorted(solves.solver_name.unique())}")
print(f"tols: {sorted(solves.orientation_tol.unique())} | seeds: {sorted(solves.seed.unique())}")
print(f"methods: {sorted(solves.generation_method.unique())}")
solves.head()

In [ ]:
(solves
 .groupby(["orientation_tol", "solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))

In [ ]:
(solves[solves.feasible]                      # only time solves that actually worked
 .groupby(["orientation_tol", "generation_method", "solver_name"])["time_median"]
 .median()
 .mul(1000).round(2)                          # ms
 .unstack("solver_name"))

In [ ]:
(solves
 .groupby(["generation_method", "orientation_tol", "solver_name"])
 .agg(feasible_rate=("feasible", "mean"),
      n=("feasible", "size"))
 .round(3))

In [ ]:
# pivot so each scenario has both solvers side by side
piv = solves.pivot_table(
    index=["seed", "orientation_tol", "scenario_id"],
    columns="solver_name",
    values=["feasible", "common_stationarity_inf", "evaluated_cost"],
    aggfunc="first")

both_feas = piv[("feasible", "SNOPT")].astype(bool) & piv[("feasible", "Ipopt")].astype(bool)
comp = piv[both_feas]
print(f"{both_feas.sum()} scenarios where both SNOPT and IPOPT feasible")

# stationarity residuals side by side, where both feasible
comp[[("common_stationarity_inf", "SNOPT"), ("common_stationarity_inf", "Ipopt")]].describe()

In [ ]:
cost_snopt = comp[("evaluated_cost", "SNOPT")]
cost_ipopt = comp[("evaluated_cost", "Ipopt")]
rel_gap = ((cost_snopt - cost_ipopt).abs() / cost_ipopt.abs())
print("relative cost gap SNOPT vs IPOPT (both feasible):")
print(rel_gap.describe())